In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
base = f"/Volumes/{catalog}/cert/raw/autoloader"

(spark.readStream.format("cloudFiles")
   .option("cloudFiles.format", "csv")
   .option("cloudFiles.schemaLocation", f"{base}/_schema")
   .option("cloudFiles.inferColumnTypes", "true")
   .load(f"{base}/input")
   .writeStream
   .option("checkpointLocation", f"{base}/_checkpoint")
   .option("mergeSchema", "true")
   .trigger(availableNow=True)
   .toTable("cert.orders_bronze")
   .awaitTermination())